# Topic Modeling Science Fiction with LDA

This notebook applies Latent Dirichlet Allocation (LDA) to the HathiTrust Extracted Features data to discover thematic clusters in science fiction. This follows Thompson & Mimno's original methodology from COLING 2018, applying it to the page-level token counts from the EF API.

Topic modeling is particularly well-suited to Extracted Features data because EF provides word frequencies without full text -- exactly what LDA needs.

## What this notebook does

1. Loads page-level token data from downloaded EF samples
2. Builds document-term matrices (one document = one volume)
3. Trains LDA models to discover thematic topics
4. Visualizes topic distributions across volumes and decades
5. Compares computational sub-genres with WWEnd/ISFDB classifications

In [ ]:
import json
import os
import glob
import numpy as np
import pandas as pd
from collections import Counter
from gensim import corpora, models
from gensim.models.coherencemodel import CoherenceModel
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load workset
scifi = pd.read_csv('data/sf-hathitrust-volumes.tsv', sep='\t')
print(f"Workset: {len(scifi)} volumes")

# Load available EF sample files
ef_files = glob.glob('data/ef_samples/*.json')
print(f"EF samples available: {len(ef_files)}")

## 1. Extract token counts from EF data

Each EF JSON file contains page-level token counts with POS tags. We aggregate tokens across all pages of each volume, filtering to content words (nouns, verbs, adjectives, adverbs) and applying a stopword list.

In [ ]:
# POS tags for content words (Penn Treebank)
CONTENT_POS = {'NN', 'NNS', 'NNP', 'NNPS',  # nouns
               'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ',  # verbs
               'JJ', 'JJR', 'JJS',  # adjectives
               'RB', 'RBR', 'RBS'}  # adverbs

# Common stopwords to remove even if tagged as content words
STOPWORDS = {'be', 'been', 'being', 'was', 'were', 'is', 'are', 'am',
             'have', 'has', 'had', 'having', 'do', 'does', 'did', 'done',
             'will', 'would', 'shall', 'should', 'may', 'might', 'can', 'could',
             'not', 'no', 'very', 'much', 'more', 'most', 'just', 'also',
             'too', 'still', 'even', 'well', 'back', 'now', 'then', 'here',
             'there', 'never', 'always', 'already', 'yet', 'ever', 'quite',
             'really', 'rather', 'perhaps', 'however', 'indeed', 'thus',
             'said', 'say', 'says', 'got', 'get', 'go', 'went', 'come',
             'came', 'make', 'made', 'take', 'took', 'know', 'knew',
             'think', 'thought', 'see', 'saw', 'look', 'looked', 'seem',
             'seemed', 'let', 'put', 'give', 'gave', 'tell', 'told',
             'thing', 'things', 'man', 'men', 'one', 'way', 'time', 'day',
             'good', 'great', 'long', 'little', 'old', 'new', 'first',
             'last', 'big', 'own', 'other', 'right', 'left'}

def extract_volume_tokens(ef_path, min_token_len=3):
    """Extract content-word token counts from an EF JSON file."""
    with open(ef_path) as f:
        data = json.load(f)

    htid = data.get('htid', os.path.basename(ef_path).replace('.json', ''))
    meta = data.get('metadata', {})
    title = meta.get('title', '?')

    token_counts = Counter()
    pages = data.get('features', {}).get('pages', [])

    for page in pages:
        body = page.get('body') or {}
        token_pos = body.get('tokenPosCount') or {}
        for token, pos_counts in token_pos.items():
            token_lower = token.lower()
            if len(token_lower) < min_token_len:
                continue
            if token_lower in STOPWORDS:
                continue
            if not token_lower.isalpha():
                continue
            # Sum counts for content POS tags only
            content_count = sum(c for pos, c in pos_counts.items() if pos in CONTENT_POS)
            if content_count > 0:
                token_counts[token_lower] += content_count

    return htid, title, token_counts

# Process all EF samples
documents = []
doc_labels = []

for ef_file in sorted(ef_files):
    htid, title, tokens = extract_volume_tokens(ef_file)
    if len(tokens) > 50:  # skip very short documents
        documents.append(tokens)
        doc_labels.append({'htid': htid, 'title': title[:60]})

print(f"Loaded {len(documents)} volumes with sufficient tokens")
for d, label in zip(documents, doc_labels):
    print(f"  {label['title']}: {sum(d.values()):,} content tokens, {len(d):,} unique")

## 2. Build dictionary and corpus

Gensim's LDA requires a dictionary (mapping tokens to IDs) and a bag-of-words corpus. We filter extremes to remove very rare and very common words.

In [ ]:
# Convert Counter objects to token lists (repeated by count, capped for efficiency)
def counter_to_tokens(counter, max_per_token=50):
    """Convert a Counter to a token list, capping repetitions."""
    tokens = []
    for tok, count in counter.items():
        tokens.extend([tok] * min(count, max_per_token))
    return tokens

token_lists = [counter_to_tokens(doc) for doc in documents]

# Build gensim dictionary
dictionary = corpora.Dictionary(token_lists)
print(f"Dictionary before filtering: {len(dictionary)} unique tokens")

# Filter: remove tokens appearing in <2 docs or >80% of docs
dictionary.filter_extremes(no_below=2, no_above=0.8)
print(f"Dictionary after filtering: {len(dictionary)} unique tokens")

# Create bag-of-words corpus
corpus = [dictionary.doc2bow(tokens) for tokens in token_lists]
print(f"Corpus: {len(corpus)} documents")
print(f"Average tokens per doc: {np.mean([sum(c for _, c in doc) for doc in corpus]):,.0f}")

## 3. Train LDA models

We train models with different numbers of topics (5, 10, 15, 20) and evaluate coherence to find the best fit. With only 22 samples, fewer topics will be more interpretable.

In [ ]:
# Train LDA models with different topic counts
topic_counts = [5, 8, 10, 15]
lda_models = {}
coherence_scores = {}

for n_topics in topic_counts:
    lda = models.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=n_topics,
        random_state=42,
        passes=15,
        alpha='auto',
        eta='auto',
        per_word_topics=True
    )
    lda_models[n_topics] = lda

    # Calculate coherence (c_v)
    cm = CoherenceModel(model=lda, texts=token_lists, dictionary=dictionary, coherence='c_v')
    score = cm.get_coherence()
    coherence_scores[n_topics] = score
    print(f"Topics={n_topics:2d}  Coherence={score:.4f}")

# Find best
best_k = max(coherence_scores, key=coherence_scores.get)
print(f"\nBest coherence: {best_k} topics ({coherence_scores[best_k]:.4f})")

## 4. Examine the best model's topics

Display the top words for each topic and attempt to label them based on science fiction sub-genre themes.

In [ ]:
# Display topics from best model
best_lda = lda_models[best_k]

print(f"=== {best_k}-Topic Model ===\n")
for idx, topic in best_lda.print_topics(num_topics=best_k, num_words=15):
    # Parse the topic string into readable format
    words = [w.split('"')[1] for w in topic.split('+')]
    weights = [float(w.split('*')[0].strip()) for w in topic.split('+')]
    print(f"Topic {idx}: {', '.join(words[:10])}")
    print(f"  (top weight: {weights[0]:.3f})")
    print()

## 5. Topic distributions per volume

Show which topics dominate each volume. This is the core of computational sub-genre classification -- volumes that share dominant topics likely belong to similar sub-genres.

In [ ]:
# Get topic distributions for each document
doc_topics = []
for i, bow in enumerate(corpus):
    topic_dist = best_lda.get_document_topics(bow, minimum_probability=0.01)
    topic_dict = {t: p for t, p in topic_dist}
    topic_dict['title'] = doc_labels[i]['title']
    topic_dict['htid'] = doc_labels[i]['htid']
    doc_topics.append(topic_dict)

topic_df = pd.DataFrame(doc_topics).fillna(0)
topic_cols = [c for c in topic_df.columns if isinstance(c, int)]

# Show dominant topic per volume
topic_df['dominant_topic'] = topic_df[topic_cols].idxmax(axis=1)
topic_df['dominant_weight'] = topic_df[topic_cols].max(axis=1)

print("Volume Topic Assignments:\n")
for _, row in topic_df.sort_values('dominant_topic').iterrows():
    print(f"  Topic {int(row['dominant_topic']):2d} ({row['dominant_weight']:.2f})  {row['title']}")

## 6. Topic heatmap

Visualize topic distributions across all volumes as a heatmap.

In [ ]:
fig, ax = plt.subplots(figsize=(12, max(6, len(documents) * 0.4)))

# Build heatmap data
heat_data = topic_df[topic_cols].values
titles = topic_df['title'].values

sns.heatmap(heat_data, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=[f'Topic {i}' for i in topic_cols],
            yticklabels=titles, ax=ax, cbar_kws={'label': 'Topic Weight'})
ax.set_title(f'Topic Distributions Across {len(documents)} Sci-Fi Volumes')
plt.tight_layout()
plt.savefig('data/topic_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to data/topic_heatmap.png")

## 7. Compare with WWEnd genre classifications

If genre data has been fetched from Worlds Without End, we can compare computational topics with human-assigned sub-genre labels.

In [ ]:
# Load WWEnd genre data if available
GENRE_FILE = 'data/wwend_genres.json'
if os.path.exists(GENRE_FILE):
    with open(GENRE_FILE) as f:
        wwend_genres = json.load(f)
    print(f"WWEnd genre data: {len(wwend_genres)} works")

    # Map htids to WWEnd IDs via the workset
    htid_to_wwend = dict(zip(scifi['htid'], scifi['WWEnd ID']))

    # Add genre info to topic_df
    genres = []
    subgenres = []
    for _, row in topic_df.iterrows():
        htid = row['htid']
        wwend_id = htid_to_wwend.get(htid)
        if wwend_id and not pd.isna(wwend_id):
            wdata = wwend_genres.get(str(int(wwend_id)), {})
            genres.append(wdata.get('genre', ''))
            subgenres.append(', '.join(wdata.get('subgenres', [])))
        else:
            genres.append('')
            subgenres.append('')

    topic_df['genre'] = genres
    topic_df['subgenres'] = subgenres

    print("\nVolume classifications:")
    for _, row in topic_df.iterrows():
        if row['genre']:
            print(f"  Topic {int(row['dominant_topic'])} | {row['genre']:15s} | {row['subgenres'][:40]:40s} | {row['title']}")
else:
    print("WWEnd genre data not yet available. Run fetch_genres.py first.")

## 8. Coherence plot and next steps

In [ ]:
# Coherence plot
fig, ax = plt.subplots(figsize=(8, 4))
ks = sorted(coherence_scores.keys())
scores = [coherence_scores[k] for k in ks]
ax.plot(ks, scores, 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('Number of Topics')
ax.set_ylabel('Coherence Score (c_v)')
ax.set_title('LDA Topic Coherence by Number of Topics')
ax.set_xticks(ks)
ax.grid(True, alpha=0.3)

# Mark the best
ax.axvline(x=best_k, color='red', linestyle='--', alpha=0.5, label=f'Best: {best_k} topics')
ax.legend()
plt.tight_layout()
plt.savefig('data/coherence_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## Next steps

- **Scale up**: Fetch EF data for more volumes (run `fetch_metadata.py`, then download EF features for a larger sample)
- **Compare sub-genres**: Once WWEnd genre data is fully fetched, cross-reference computational topics with human classifications
- **Temporal analysis**: Track how topic distributions shift across decades (1900s-2010s)
- **Author signatures**: Compare topic profiles across prolific authors (Wells, Heinlein, Silverberg)
- **Anthology segmentation**: Apply topic modeling at the page level to identify distinct stories within anthology volumes